# Verificador por lotes para organizadores (115696)
Versión mejorada del verificador de gatos: procesa una carpeta completa, usa un umbral de confianza configurable, genera un informe CSV y un resumen final. Sube `keras_model.h5`, `labels.txt` y una carpeta `fotos/`.

In [ ]:
from google.colab import files

files.upload()  # keras_model.h5 y labels.txt

## Clasificador (igual que en el verificador simple)

In [ ]:
import os

import numpy as np
from PIL import Image, ImageOps

_modelo = None  # caché para no recargar el modelo en cada foto


def get_class(image_path, model_path, labels_path):
    """Devuelve (clase, probabilidad). Lanza FileNotFoundError si falta algún archivo."""
    global _modelo
    for ruta in (image_path, model_path, labels_path):
        if not os.path.isfile(ruta):
            raise FileNotFoundError(f"No existe el archivo: {ruta}")

    if _modelo is None:
        from keras.models import load_model  # import tardío: TensorFlow es pesado

        _modelo = load_model(model_path, compile=False)
    with open(labels_path, "r", encoding="utf-8") as f:
        class_names = f.readlines()

    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    image = Image.open(image_path).convert("RGB")
    image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)
    data[0] = (np.asarray(image).astype(np.float32) / 127.5) - 1

    prediction = _modelo.predict(data)
    index = int(np.argmax(prediction))
    return class_names[index][2:].strip(), float(prediction[0][index])


def verificar_foto(image_path, model_path="keras_model.h5", labels_path="labels.txt"):
    """Mensaje para el participante según la clase detectada."""
    try:
        clase, prob = get_class(image_path, model_path, labels_path)
    except FileNotFoundError as error:
        return f"Error: {error}"
    except Exception as error:  # imagen corrupta o modelo inválido
        return f"Error al procesar la foto: {error}"

    if clase == "Real":
        return f"Foto ACEPTADA: parece una fotografía real ({prob:.1%})."
    return f"Foto RECHAZADA: parece generada por IA ({prob:.1%}). El concurso solo admite fotos reales."

## Procesamiento de la carpeta

In [ ]:
import csv
import os


EXTENSIONES = (".jpg", ".jpeg", ".png", ".webp")


def decidir(clase, prob, umbral):
    """ACEPTADA solo si es Real con confianza suficiente; si no hay confianza, REVISAR a mano."""
    if prob < umbral:
        return "REVISAR"
    return "ACEPTADA" if clase == "Real" else "RECHAZADA"


def procesar_carpeta(carpeta, model_path, labels_path, umbral=0.8, informe="informe.csv", clasificar=get_class):
    archivos = sorted(f for f in os.listdir(carpeta) if f.lower().endswith(EXTENSIONES))
    filas = []
    for nombre in archivos:
        try:
            clase, prob = clasificar(os.path.join(carpeta, nombre), model_path, labels_path)
            filas.append([nombre, clase, f"{prob:.4f}", decidir(clase, prob, umbral)])
        except FileNotFoundError:
            raise  # modelo o etiquetas ausentes: no tiene sentido seguir
        except Exception as error:
            filas.append([nombre, "", "", f"ERROR: {error}"])

    with open(informe, "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["archivo", "clase", "probabilidad", "decision"])
        escritor.writerows(filas)

    resumen = {}
    for fila in filas:
        clave = fila[3].split(":")[0]
        resumen[clave] = resumen.get(clave, 0) + 1
    return filas, resumen


## Ejecutar
Cambia `UMBRAL` para ser más o menos estricto.

In [ ]:
UMBRAL = 0.8

filas, resumen = procesar_carpeta('fotos', 'keras_model.h5', 'labels.txt', UMBRAL, 'informe.csv')
print(f'Fotos analizadas: {len(filas)} (umbral {UMBRAL:.0%})')
for clave, total in resumen.items():
    print(f'  {clave}: {total}')
import pandas as pd
pd.read_csv('informe.csv')